# Lead Scoring via Bagging Positive-Unlabeled (PU) Learning

**Goal.** Rank ~27,854 Tunisian businesses scraped from Google Maps by
their likelihood of becoming Ooredoo Business customers, given only ~595
known positives (matched via entity linkage) and an unlabeled remainder.

**Method.** Bagging-PU (Mordelet & Vert, 2014) with LightGBM as the base
classifier. Compared against a naive baseline that treats all unlabeled
as negative.

**Validation.** 20% of known positives are held out before training and
used to measure recovery via Precision@k and Recall@k.

**Outputs.** `ranked_prospects.csv` (all businesses scored & ranked).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED       = 42
N_ITERATIONS      = 50           # PU bagging rounds. 50 = a solid start; 100 = smoother, ~2x slower.
NEGATIVE_MULTIPLE = 3            # sample this-many pseudo-negatives per known positive each round.
HOLDOUT_FRAC      = 0.20         # fraction of positives held out for validation.
K_VALUES          = [100, 500, 1000, 2000, 5000]

rng = np.random.RandomState(RANDOM_SEED)

## 2. Load features

In [ ]:
features = pd.read_csv("features.csv")

# Re-cast categoricals -- CSV round-trip loses the 'category' dtype.
for c in ("category", "search_category", "governorate"):
    features[c] = features[c].astype("category")

y = features["is_customer"].values
X = features.drop(columns=["is_customer"])

print(f"Rows            : {len(X):,}")
print(f"Features        : {X.shape[1]}")
print(f"Known positives : {int(y.sum()):,}  ({y.mean()*100:.2f}%)")
print(f"Unlabeled       : {int((y == 0).sum()):,}")

## 3. Train / held-out split

We hide 20% of the known positives from training and treat them as
"unlabeled" during modeling. After training, we check whether the model
ranks these hidden positives near the top -- that's our recall proxy.

In [ ]:
pos_idx = np.where(y == 1)[0]
unl_idx = np.where(y == 0)[0]

train_pos, holdout_pos = train_test_split(
    pos_idx, test_size=HOLDOUT_FRAC, random_state=RANDOM_SEED
)

# Training labels: 1 for train_pos, 0 for everything else (including holdout).
y_train = np.zeros(len(y), dtype=int)
y_train[train_pos] = 1

# Held-out positives are marked so we can evaluate on them later.
is_holdout = np.zeros(len(y), dtype=bool)
is_holdout[holdout_pos] = True

print(f"Training positives : {len(train_pos):,}")
print(f"Held-out positives : {len(holdout_pos):,}   (used only for evaluation)")
print(f"Unlabeled pool     : {(y_train == 0).sum() - len(holdout_pos):,}"
      f"  (plus {len(holdout_pos)} disguised held-out positives)")

## 4. Baseline: naive classifier (unlabeled = negative)

The straw-man from the thesis narrative. Trains one LightGBM with the
whole unlabeled pool as negative. We *expect* this to be biased -- its
scores are dragged toward 0 because ~2% of the "negatives" are actually
hidden positives, teaching the model to distrust customer-like patterns.

In [ ]:
def make_lgb() -> lgb.LGBMClassifier:
    """Fresh classifier with sensible defaults for this dataset."""
    return lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=31,
        min_child_samples=20, random_state=RANDOM_SEED,
        n_jobs=-1, verbose=-1,
    )

naive_clf = make_lgb()
naive_clf.fit(X, y_train)
naive_scores = naive_clf.predict_proba(X)[:, 1]

print(f"Naive baseline trained. Score range: [{naive_scores.min():.4f}, {naive_scores.max():.4f}]")

## 5. Bagging-PU

For each of N_ITERATIONS rounds:
  * Sample K unlabeled points as pseudo-negatives (K = NEGATIVE_MULTIPLE × #positives).
  * Train a LightGBM on {training positives} vs {this round's pseudo-negatives}.
  * Score the points *not* used as pseudo-negatives this round (out-of-bag).

Each point's final score is the average across all rounds it was OOB.
OOB scoring avoids the leak of scoring a point on a model that just saw
it labeled as negative.

In [ ]:
n_pos_train = len(train_pos)
neg_sample_size = NEGATIVE_MULTIPLE * n_pos_train

score_sum   = np.zeros(len(X))
score_count = np.zeros(len(X), dtype=int)

# Unlabeled pool (for sampling pseudo-negatives) excludes training positives
# but INCLUDES held-out positives -- they're supposed to look like unknowns.
unlabeled_pool = np.where(y_train == 0)[0]

print(f"Running {N_ITERATIONS} PU iterations, {neg_sample_size:,} pseudo-negatives each...")
for i in range(N_ITERATIONS):
    neg_sample = rng.choice(unlabeled_pool, size=neg_sample_size, replace=False)
    train_idx  = np.concatenate([train_pos, neg_sample])
    y_round    = np.concatenate([np.ones(n_pos_train), np.zeros(neg_sample_size)]).astype(int)

    clf = make_lgb()
    clf.fit(X.iloc[train_idx], y_round)

    # OOB = every row NOT used as a pseudo-negative this round.
    oob_mask = np.ones(len(X), dtype=bool)
    oob_mask[neg_sample] = False
    oob_scores = clf.predict_proba(X[oob_mask])[:, 1]

    score_sum[oob_mask]   += oob_scores
    score_count[oob_mask] += 1

    if (i + 1) % 10 == 0:
        print(f"  iter {i+1:3d}/{N_ITERATIONS}")

# Average -- every point should have been OOB many times.
assert score_count.min() > 0, "some point never scored; increase iterations"
pu_scores = score_sum / score_count
print(f"\nBagging-PU done. Score range: [{pu_scores.min():.4f}, {pu_scores.max():.4f}]")
print(f"Each point averaged over {score_count.mean():.1f} rounds (min {score_count.min()}, max {score_count.max()}).")

## 6. Evaluation: Precision@k and Recall@k on held-out positives

We hid HOLDOUT_FRAC of the true positives. A good model ranks them near
the top of the full score list -- ideally within the top few thousand,
so Ooredoo sales can call those first.

In [ ]:
def precision_recall_at_k(scores: np.ndarray, is_target: np.ndarray, ks: list[int]) -> pd.DataFrame:
    """Rank all rows by score desc; for each k, report Precision@k and Recall@k
    against the target mask (True = held-out positive)."""
    order = np.argsort(-scores)
    hits_cum = np.cumsum(is_target[order])
    total_targets = is_target.sum()
    rows = []
    for k in ks:
        hits = int(hits_cum[k - 1])
        rows.append({
            "k": k, "hits": hits,
            "precision@k": hits / k,
            "recall@k": hits / total_targets,
        })
    return pd.DataFrame(rows)

naive_metrics = precision_recall_at_k(naive_scores, is_holdout, K_VALUES)
pu_metrics    = precision_recall_at_k(pu_scores,    is_holdout, K_VALUES)

print("=== Naive baseline ===")
print(naive_metrics.to_string(index=False))
print("\n=== Bagging-PU ===")
print(pu_metrics.to_string(index=False))

# Improvement side-by-side
comparison = pd.DataFrame({
    "k": K_VALUES,
    "naive_recall@k": naive_metrics["recall@k"].values,
    "pu_recall@k":    pu_metrics["recall@k"].values,
    "recall_lift":    pu_metrics["recall@k"].values - naive_metrics["recall@k"].values,
})
print("\n=== Recall lift (PU - naive) ===")
print(comparison.round(3).to_string(index=False))

## 7. Plots

Two views: how held-out positives are distributed in the score space
(should shift right vs random unlabeled), and the recall curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: score histogram, held-out positives vs random unlabeled sample.
sample_unl = rng.choice(np.where(~is_holdout & (y_train == 0))[0], size=5000, replace=False)
axes[0].hist(pu_scores[sample_unl],   bins=40, alpha=0.5, label="unlabeled (sample)", density=True)
axes[0].hist(pu_scores[is_holdout],   bins=40, alpha=0.7, label="held-out positives", density=True)
axes[0].set_xlabel("PU score")
axes[0].set_ylabel("density")
axes[0].set_title("Bagging-PU: held-out positives shift right")
axes[0].legend()

# Right: recall@k, naive vs PU.
ks_smooth = list(range(50, 6001, 50))
naive_curve = precision_recall_at_k(naive_scores, is_holdout, ks_smooth)
pu_curve    = precision_recall_at_k(pu_scores,    is_holdout, ks_smooth)
axes[1].plot(ks_smooth, naive_curve["recall@k"], label="Naive baseline")
axes[1].plot(ks_smooth, pu_curve["recall@k"],    label="Bagging-PU")
axes[1].set_xlabel("k (top-ranked prospects)")
axes[1].set_ylabel("Recall@k on held-out positives")
axes[1].set_title("Recall curve")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("evaluation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: evaluation.png")

## 8. Feature importance (SHAP on one representative model)

We train one classifier on the full training positives + one random
negative sample and use its SHAP values as a stand-in for global
feature importance. Cheaper than SHAP-per-iteration and directionally
identical for well-averaged bagging.

In [ ]:
import shap

neg_sample = rng.choice(unlabeled_pool, size=neg_sample_size, replace=False)
train_idx  = np.concatenate([train_pos, neg_sample])
y_round    = np.concatenate([np.ones(n_pos_train), np.zeros(neg_sample_size)]).astype(int)

representative_clf = make_lgb()
representative_clf.fit(X.iloc[train_idx], y_round)

explainer = shap.TreeExplainer(representative_clf)
# Sample for speed -- SHAP on 27k rows is slow.
shap_sample_idx = rng.choice(len(X), size=2000, replace=False)
shap_values = explainer.shap_values(X.iloc[shap_sample_idx])

shap.summary_plot(shap_values, X.iloc[shap_sample_idx], plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: feature_importance.png")

## 9. Final deliverable: ranked prospect list

`ranked_prospects.csv` has every business scored & sorted, joined back
to the identifying columns from `maps_labeled.csv`. This is what Ooredoo
sales would work top-down.

In [ ]:
maps = pd.read_csv("maps_labeled.csv")
assert len(maps) == len(pu_scores), "row count mismatch -- maps_labeled and features must be in the same order"

output = maps.copy()
output["pu_score"]        = pu_scores
output["naive_score"]     = naive_scores
output["is_known_customer"] = output["is_customer"]      # rename for clarity
output = output.drop(columns=["is_customer"])
output = output.sort_values("pu_score", ascending=False).reset_index(drop=True)
output.insert(0, "rank", np.arange(1, len(output) + 1))

output.to_csv("ranked_prospects.csv", index=False)
print(f"Saved: ranked_prospects.csv  ({len(output):,} rows)")
print("\n=== Top 10 prospects (excluding known customers) ===")
top10 = output[output["is_known_customer"] == 0].head(10)
print(top10[["rank", "name", "governorate", "category", "pu_score"]].to_string(index=False))